<a href="https://colab.research.google.com/github/gdhameja1/ML/blob/ML_ALgo/Healthcare_Patient_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCA on Healthcare Patient Metrics

This notebook performs Principal Component Analysis (PCA) on the `Healthcare_Patient_Metrics_100.csv` file.

**Steps covered:**
1. Load the data and inspect structure
2. Drop `Patient_ID` and retain numeric metric columns
3. Handle missing values (median imputation by default)
4. Standardize variables (z-scores)
5. Run PCA on standardized data
6. Inspect variance explained by each principal component
7. Examine loadings for top components (PC1–PC3)
8. Save detailed results to an Excel file for further analysis in Excel


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option('display.float_format', lambda x: f'{x:0.4f}')

## 1. Load the data

Make sure the file `Healthcare_Patient_Metrics_100.csv` is in the same folder as this notebook.

In [ ]:
CSV_FILE = "Healthcare_Patient_Metrics_100.csv"
ID_COL = "Patient_ID"  # first column in the file

# Load CSV
df = pd.read_csv(CSV_FILE)
print("Data shape:", df.shape)
df.head()

Data shape: (100, 22)


,Patient_ID,Systolic_BP,Diastolic_BP,Total_Cholesterol,LDL_Cholesterol,HDL_Cholesterol,Heart_Rate,Fasting_Glucose,HbA1c,BMI,...,CRP,WBC_Count,ESR,Creatinine,BUN,eGFR,ALT,AST,Hemoglobin,Platelet_Count
0,P001,119.5000,87.7000,222.5000,128.2000,50.7000,77.4000,73.7000,4.1000,19.6000,...,2.9300,7.5700,18.9000,0.9400,12.2000,99.1000,15.6000,25.3000,15.6000,214.0000
1,P002,114.9000,84.3000,186.6000,124.9000,48.5000,69.3000,91.2000,4.1300,22.5000,...,2.5700,9.2200,21.0000,0.8300,11.5000,97.7000,17.1000,24.3000,12.5000,259.0000
2,P003,129.7000,82.3000,228.1000,130.6000,45.3000,77.3000,88.4000,5.7200,23.9000,...,3.2200,8.6500,25.6000,1.1500,21.1000,74.8000,17.9000,28.5000,13.4000,223.0000
3,P004,143.1000,96.9000,259.2000,146.8000,35.5000,89.3000,75.7000,4.5000,22.9000,...,3.5800,8.4700,26.8000,1.0300,18.8000,79.3000,24.2000,17.2000,14.4000,236.0000
4,P005,114.2000,75.7000,197.1000,101.7000,49.7000,70.9000,95.3000,5.6400,24.5000,...,0.1000,4.0000,5.2000,0.8100,14.7000,80.8000,17.8000,23.0000,17.0000,334.0000


## 2. Prepare feature matrix (drop `Patient_ID`)
We keep only numeric metric columns and exclude the patient identifier.

In [ ]:
if ID_COL not in df.columns:
    raise ValueError(f"Expected ID column '{ID_COL}' not found. Columns: {list(df.columns)}")

patient_id = df[ID_COL]

# Keep only numeric columns (excluding ID)
X = df.drop(columns=[ID_COL])
X = X.select_dtypes(include=[np.number])

feature_names = X.columns.tolist()
print("Number of features used in PCA:", len(feature_names))
X.head()

Number of features used in PCA: 21


,Systolic_BP,Diastolic_BP,Total_Cholesterol,LDL_Cholesterol,HDL_Cholesterol,Heart_Rate,Fasting_Glucose,HbA1c,BMI,Triglycerides,...,CRP,WBC_Count,ESR,Creatinine,BUN,eGFR,ALT,AST,Hemoglobin,Platelet_Count
0,119.5000,87.7000,222.5000,128.2000,50.7000,77.4000,73.7000,4.1000,19.6000,83.4000,...,2.9300,7.5700,18.9000,0.9400,12.2000,99.1000,15.6000,25.3000,15.6000,214.0000
1,114.9000,84.3000,186.6000,124.9000,48.5000,69.3000,91.2000,4.1300,22.5000,148.1000,...,2.5700,9.2200,21.0000,0.8300,11.5000,97.7000,17.1000,24.3000,12.5000,259.0000
2,129.7000,82.3000,228.1000,130.6000,45.3000,77.3000,88.4000,5.7200,23.9000,133.7000,...,3.2200,8.6500,25.6000,1.1500,21.1000,74.8000,17.9000,28.5000,13.4000,223.0000
3,143.1000,96.9000,259.2000,146.8000,35.5000,89.3000,75.7000,4.5000,22.9000,106.6000,...,3.5800,8.4700,26.8000,1.0300,18.8000,79.3000,24.2000,17.2000,14.4000,236.0000
4,114.2000,75.7000,197.1000,101.7000,49.7000,70.9000,95.3000,5.6400,24.5000,151.6000,...,0.1000,4.0000,5.2000,0.8100,14.7000,80.8000,17.8000,23.0000,17.0000,334.0000


## 3. Handle missing values

For simplicity, we **impute missing values with the median** of each column. You can change this strategy if needed.

In [ ]:
# Option A: Drop rows with any missing values (commented out)
# X = X.dropna()
# patient_id = patient_id.loc[X.index]

# Option B: Median imputation (used here)
X = X.fillna(X.median(numeric_only=True))

X.isna().sum()

,0
Systolic_BP,0
Diastolic_BP,0
Total_Cholesterol,0
LDL_Cholesterol,0
HDL_Cholesterol,0
Heart_Rate,0
Fasting_Glucose,0
HbA1c,0
BMI,0
Triglycerides,0


## 4. Standardize variables (z-scores)

We standardize each metric so that it has mean 0 and standard deviation 1. This ensures variables with larger scales do not dominate PCA.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names)
X_scaled_df.head()

,Systolic_BP,Diastolic_BP,Total_Cholesterol,LDL_Cholesterol,HDL_Cholesterol,Heart_Rate,Fasting_Glucose,HbA1c,BMI,Triglycerides,...,CRP,WBC_Count,ESR,Creatinine,BUN,eGFR,ALT,AST,Hemoglobin,Platelet_Count
0,0.0910,1.0176,0.8667,0.4721,-0.1406,0.7302,-1.1904,-1.3253,-1.3132,-1.6231,...,0.4978,0.1002,0.3304,-0.3340,-0.7604,0.8185,-1.0765,-0.3726,1.1553,-0.8936
1,-0.2458,0.6331,-0.3655,0.3284,-0.3840,-0.2380,-0.2795,-1.2988,-0.6397,-0.0549,...,0.2697,0.9109,0.5737,-0.7399,-0.9102,0.7168,-0.9080,-0.4960,-1.0501,0.1018
2,0.8379,0.4070,1.0589,0.5766,-0.7380,0.7182,-0.4252,0.1079,-0.3145,-0.4040,...,0.6816,0.6308,1.1066,0.4410,1.1443,-0.9473,-0.8181,0.0226,-0.4098,-0.6946
3,1.8191,2.0580,2.1263,1.2820,-1.8222,2.1525,-1.0863,-0.9714,-0.5468,-1.0608,...,0.9097,0.5424,1.2457,-0.0018,0.6521,-0.6203,-0.1101,-1.3728,0.3016,-0.4070
4,-0.2971,-0.3394,-0.0051,-0.6818,-0.2512,-0.0467,-0.0661,0.0372,-0.1751,0.0299,...,-1.2955,-1.6539,-1.2568,-0.8137,-0.2254,-0.5113,-0.8293,-0.6566,2.1513,1.7607


## 5. Run PCA

We run PCA on the standardized data and inspect how much variance each principal component explains.

In [ ]:
pca = PCA()  # keep all components
scores = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
csr = np.cumsum(explained)

pca_summary = pd.DataFrame({
    "PC": [f"PC{i}" for i in range(1, len(explained) + 1)],
    "Eigenvalue": pca.explained_variance_,
    "Variance_Explained": explained,
    "Cumulative_Variance": csr
})

pca_summary.head(10)

,PC,Eigenvalue,Variance_Explained,Cumulative_Variance
0,PC1,6.5190,0.3073,0.3073
1,PC2,5.1202,0.2414,0.5487
2,PC3,2.9121,0.1373,0.6860
3,PC4,2.5249,0.1190,0.8050
4,PC5,1.0984,0.0518,0.8568
5,PC6,0.9246,0.0436,0.9004
6,PC7,0.3155,0.0149,0.9153
7,PC8,0.2512,0.0118,0.9271
8,PC9,0.2306,0.0109,0.9380
9,PC10,0.1951,0.0092,0.9472


In [ ]:
print("Explained variance ratio (first 10 PCs):")
for i, v in enumerate(explained[:10], start=1):
    print(f"PC{i}: {v:.4f}  (cumulative: {csr[i-1]:.4f})")

Explained variance ratio (first 10 PCs):
PC1: 0.3073  (cumulative: 0.3073)
PC2: 0.2414  (cumulative: 0.5487)
PC3: 0.1373  (cumulative: 0.6860)
PC4: 0.1190  (cumulative: 0.8050)
PC5: 0.0518  (cumulative: 0.8568)
PC6: 0.0436  (cumulative: 0.9004)
PC7: 0.0149  (cumulative: 0.9153)
PC8: 0.0118  (cumulative: 0.9271)
PC9: 0.0109  (cumulative: 0.9380)
PC10: 0.0092  (cumulative: 0.9472)


## 6. Component loadings (contribution of each variable to each PC)

The loading of a variable on a principal component tells you how strongly that variable contributes to the component.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=[f"PC{i}" for i in range(1, len(feature_names) + 1)]
)

loadings.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21
Systolic_BP,-0.2027,0.3273,-0.0284,-0.1537,-0.0010,-0.0784,0.2870,0.3133,0.0136,0.1369,...,-0.1012,-0.3558,0.4310,-0.2157,-0.4232,-0.1138,-0.0583,-0.0862,0.1090,0.0270
Diastolic_BP,-0.2430,0.3089,-0.0516,-0.1310,-0.0035,0.0469,0.0469,0.0841,-0.1551,0.2676,...,-0.0995,-0.0538,-0.4217,0.0422,0.0903,0.1443,0.3390,0.0412,0.4772,-0.3064
Total_Cholesterol,-0.2099,0.3267,-0.0431,-0.1485,0.0426,-0.1131,0.0506,0.2109,0.2763,0.1004,...,0.0125,0.1799,-0.0189,0.6526,0.0831,0.2144,0.0017,0.0547,-0.3511,0.0903
LDL_Cholesterol,-0.2212,0.3351,-0.0497,-0.0924,-0.0786,0.0432,0.0138,-0.0160,-0.0166,0.1506,...,0.0903,0.0530,-0.2235,-0.4285,0.4918,-0.1357,-0.4810,-0.0987,-0.1759,0.1026
HDL_Cholesterol,0.2369,-0.2906,-0.0568,0.1092,-0.0145,-0.0865,0.1562,0.2260,0.5562,0.5327,...,0.0174,0.1519,-0.1274,-0.2368,-0.0048,-0.0151,0.0692,0.1105,0.0053,-0.1795


### Top contributing variables for the first 3 principal components

In [ ]:
TOP_N = 8

for pc in ["PC1", "PC2", "PC3"]:
    top = loadings[pc].abs().sort_values(ascending=False).head(TOP_N)
    print("\n", "-"*60)
    print(f"Top {TOP_N} variables by |loading| for {pc}:")
    display(pd.DataFrame({
        "Variable": top.index,
        "Loading": loadings.loc[top.index, pc].values
    }))


 ------------------------------------------------------------
Top 8 variables by |loading| for PC1:


,Variable,Loading
0,Fasting_Glucose,0.3403
1,Triglycerides,0.3255
2,BMI,0.3224
3,HbA1c,0.3168
4,Waist_Circumference,0.3057
5,AST,0.3016
6,ALT,0.2817
7,Diastolic_BP,-0.2430



 ------------------------------------------------------------
Top 8 variables by |loading| for PC2:


,Variable,Loading
0,LDL_Cholesterol,0.3351
1,Systolic_BP,0.3273
2,Total_Cholesterol,0.3267
3,Heart_Rate,0.3252
4,Diastolic_BP,0.3089
5,HDL_Cholesterol,-0.2906
6,HbA1c,0.2277
7,AST,0.2190



 ------------------------------------------------------------
Top 8 variables by |loading| for PC3:


,Variable,Loading
0,Creatinine,0.4219
1,eGFR,-0.4050
2,ESR,0.3962
3,BUN,0.3930
4,WBC_Count,0.3901
5,CRP,0.3885
6,Platelet_Count,-0.1554
7,Hemoglobin,0.0840


## 7. Patient scores on the first 3 principal components

Each patient receives a score on each principal component, which can be thought of as an index summarizing their health along different dimensions (e.g., Metabolic Risk, Cardiovascular Risk, Renal-Inflammatory Burden).

In [ ]:
scores_df = pd.DataFrame(scores[:, :3], columns=["PC1_Score", "PC2_Score", "PC3_Score"])
scores_df.insert(0, ID_COL, patient_id.values)

scores_df.head()

,Patient_ID,PC1_Score,PC2_Score,PC3_Score
0,P001,-3.6194,-0.4064,-0.3825
1,P002,-1.4848,-0.2154,-0.4145
2,P003,-1.5870,1.0922,1.9354
3,P004,-4.4696,2.6231,1.3939
4,P005,-0.0382,-1.2318,-1.8963


## 8. Save key outputs to Excel

This creates an Excel file with:
- Raw data
- Standardized z-scores
- Correlation matrix
- PCA variance summary
- Loadings
- Top 3 PC scores for each patient

In [ ]:
OUT_XLSX = "Healthcare_PCA_Results.xlsx"

with pd.ExcelWriter(OUT_XLSX) as writer:
    df.to_excel(writer, sheet_name="Raw_Data", index=False)
    X_scaled_df.assign(Patient_ID=patient_id.values).set_index("Patient_ID").to_excel(writer, sheet_name="Standardized_Z")
    df[feature_names].corr().to_excel(writer, sheet_name="Correlation_Matrix")
    pca_summary.to_excel(writer, sheet_name="PCA_Variance_Summary", index=False)
    loadings.to_excel(writer, sheet_name="Loadings")
    scores_df.to_excel(writer, sheet_name="Top3_PC_Scores", index=False)

print(f"Saved PCA outputs to: {OUT_XLSX}")

Saved PCA outputs to: Healthcare_PCA_Results.xlsx
